<a href="https://colab.research.google.com/github/pesco13/Python-para-Quimica/blob/main/notebooks/11_Informe_reproducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 11 · El informe final reproducible

**Objetivos**
* Organizar el trabajo de todo el curso en una estructura de carpetas y nombres que se entienda sola.
* Dejar registrado el entorno (qué versiones se usaron) para que el cálculo se pueda repetir.
* Dar a todas las figuras del informe el mismo aspecto.
* Verificar con código que el portafolio está completo antes de entregarlo.

Un resultado que nadie puede repetir no es un resultado. Esta práctica recoge lo que hace falta para que un trabajo propio quede en condiciones de ser repetido por otra persona sin preguntarle nada a usted.

## 1. Qué significa que un trabajo sea reproducible

Tres niveles, de menos a más exigente:

| Nivel | Qué permite | Qué hace falta |
|---|---|---|
| Repetible | Usted vuelve a correrlo y da lo mismo | Guardar el notebook y los datos |
| Reproducible | Otra persona lo corre y da lo mismo | Además: datos accesibles, versiones anotadas, orden de ejecución claro |
| Replicable | Otra persona lo rehace desde cero y llega a la misma conclusión | Además: método descrito con suficiente detalle |

El objetivo razonable para un trabajo propio es el nivel intermedio. Los dos enemigos habituales son el notebook que solo corre si se ejecutan las celdas en un orden secreto y el archivo de datos que estaba "en el escritorio".

## 2. Estructura de carpetas

Una estructura que funciona para cualquier trabajo pequeño:

```
mi_informe/
├── datos/              datos de entrada, tal como llegaron, sin modificar
│   ├── calibracion_patrones.csv
│   └── cinetica.csv
├── resultados/         lo que produce el código
│   ├── tabla_descriptores.csv
│   └── figuras/
│       ├── fig01_calibracion.png
│       └── fig02_arrhenius.png
├── informe.ipynb       el notebook que va de datos/ a resultados/
└── README.md           qué es esto, quién lo hizo, cómo se corre
```

La regla de oro: **nunca se modifica un archivo de `datos/`**. Si hay que corregir algo, el código lo corrige y escribe el resultado en `resultados/`. Así siempre se puede volver al punto de partida.

In [ ]:
import os

CARPETAS = ["mi_informe/datos", "mi_informe/resultados/figuras"]
for carpeta in CARPETAS:
    os.makedirs(carpeta, exist_ok=True)

print("Estructura creada:")
for raiz, _, archivos in os.walk("mi_informe"):
    print(" ", raiz)

### Nombres de archivo

Tres reglas que ahorran horas:

* **Sin espacios ni acentos.** `fig01_calibracion.png`, no `Figura 1 calibración.png`. Los espacios rompen comandos y los acentos se corrompen al pasar de un sistema a otro.
* **Con número delante** cuando el orden importa: `fig01`, `fig02`. Así se ordenan solos.
* **Fechas al revés**: `2026-03-15`, no `15-03-2026`. Ordenadas alfabéticamente quedan en orden cronológico.

## 3. Registrar el entorno

Dentro de un año, `scipy` habrá cambiado. Si no anotó qué versión usó, no podrá explicar por qué el resultado ya no sale igual. Esta celda deja constancia.

In [ ]:
import sys, platform
from datetime import date

import numpy, pandas, scipy, matplotlib

def registro_de_entorno():
    lineas = [
        f"Fecha de ejecucion : {date.today().isoformat()}",
        f"Python             : {sys.version.split()[0]} ({platform.system()})",
        f"numpy              : {numpy.__version__}",
        f"pandas             : {pandas.__version__}",
        f"scipy              : {scipy.__version__}",
        f"matplotlib         : {matplotlib.__version__}",
    ]
    try:
        import rdkit
        lineas.append(f"rdkit              : {rdkit.__version__}")
    except ImportError:
        pass
    return "\n".join(lineas)

registro = registro_de_entorno()
print(registro)

with open("mi_informe/resultados/entorno.txt", "w", encoding="utf-8") as f:
    f.write(registro + "\n")

Ese archivo `entorno.txt` se entrega junto con el informe. Es la diferencia entre "me salía bien" y una afirmación verificable.

## 4. Un estilo común para todas las figuras

Si cada figura tiene un tamaño de letra distinto, el informe se ve descuidado. `matplotlib` permite fijar los valores por defecto una sola vez, al principio del notebook y a partir de ahí todas las figuras salen iguales.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (6, 4),
    "figure.dpi": 110,          # en pantalla
    "savefig.dpi": 300,         # al guardar
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "axes.spines.top": False,   # quita los bordes superior y derecho
    "axes.spines.right": False,
    "legend.frameon": False,
    "lines.markersize": 5,
})

print("Estilo aplicado. Todas las figuras siguientes lo heredan.")

In [ ]:
import numpy as np

def guardar_figura(fig, numero, nombre):
    """Guarda la figura con nombre normalizado dentro de resultados/figuras."""
    ruta = f"mi_informe/resultados/figuras/fig{numero:02d}_{nombre}.png"
    fig.savefig(ruta)
    return ruta

x = np.linspace(0, 10, 40)
fig, ax = plt.subplots()
ax.plot(x, 0.085 * x + 0.004, "-", color="#1b6ca8")
ax.set_xlabel("Concentración (mg L$^{-1}$)")
ax.set_ylabel("Absorbancia")
ax.set_title("Curva de calibración del analito asignado")

print("Guardada en:", guardar_figura(fig, 1, "calibracion"))
plt.show()

## 5. Cómo se escribe la sección de métodos

Un método reproducible responde a cuatro preguntas, en este orden: **qué datos**, **qué se les hizo**, **con qué herramienta** y **qué parámetros**. Comparemos:

> Se ajustaron los datos y se obtuvo la energía de activación.

No permite repetir nada. Frente a:

> La concentración frente al tiempo se ajustó al modelo integrado de segundo orden mediante mínimos cuadrados no lineales (`scipy.optimize.curve_fit`, SciPy 1.14.1), con estimación inicial `[A]₀ = 0,08 mol L⁻¹` y `k = 0,02`. Las incertidumbres de los parámetros se tomaron como la raíz de la diagonal de la matriz de covarianza. La energía de activación se obtuvo de la pendiente de ln k frente a 1/T sobre las cuatro temperaturas medidas.

Esa segunda versión la puede repetir cualquiera. Fíjese en que **no** describe el código línea por línea: describe las decisiones.

## 6. La plantilla del informe

Un informe corto sigue la estructura de un artículo:

1. **Portada**: título, autor, afiliación y fecha.
2. **Resumen**: cinco líneas con lo que hizo y el resultado principal, con su número.
3. **Introducción**: qué sistema químico le tocó y por qué tiene interés.
4. **Métodos**: según el punto 5, incluyendo el origen de los datos.
5. **Resultados**: las figuras y tablas, numeradas y con pie.
6. **Discusión**: qué significan los números y qué limitaciones tienen.
7. **Conclusiones**: tres o cuatro frases.
8. **Referencias**: bibliotecas usadas y fuentes de datos.
9. **Anexo**: el registro del entorno.

La celda siguiente escribe el esqueleto en un archivo Markdown para que lo rellene.

In [ ]:
plantilla = """# Título del informe

**Autor** · **Fecha**

## Resumen

(Cinco líneas. Qué hizo, con qué datos y cuál fue el resultado principal con su incertidumbre.)

## Introducción

(Qué familia de compuestos le tocó y por qué importa.)

## Métodos

(Datos de partida, tratamiento, bibliotecas con versión, parámetros.)

## Resultados

**Figura 1.** Curva de calibración ...

**Tabla 1.** Descriptores moleculares ...

## Discusión

(Qué significan los números. Qué supuestos se hicieron. Qué no se puede concluir.)

## Conclusiones

## Referencias

1. Virtanen, P. et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261-272.
2. Landrum, G. RDKit: Open-source cheminformatics. https://www.rdkit.org
3. Kim, S. et al. (2023). PubChem 2023 update. *Nucleic Acids Res.*, 51, D1373-D1380.

## Anexo: entorno de cálculo

(Pegue aquí el contenido de entorno.txt)
"""

with open("mi_informe/plantilla_informe.md", "w", encoding="utf-8") as f:
    f.write(plantilla)

print("Plantilla escrita en mi_informe/plantilla_informe.md")

## 7. Verificar el portafolio antes de entregar

En vez de revisar a mano si están todos los archivos, que lo haga el código.

In [ ]:
ESPERADOS = [
    "mi_informe/plantilla_informe.md",
    "mi_informe/resultados/entorno.txt",
    "mi_informe/resultados/figuras/fig01_calibracion.png",
]

def revisar_portafolio(esperados):
    faltan = [r for r in esperados if not os.path.exists(r)]
    for r in esperados:
        print(("  ok   " if r not in faltan else "  FALTA"), r)
    return faltan

faltan = revisar_portafolio(ESPERADOS)
print()
print("Portafolio completo." if not faltan else f"Faltan {len(faltan)} archivos.")

Amplíe la lista `ESPERADOS` con los archivos que debe contener su trabajo y corra la celda antes de darlo por terminado. Es la misma idea de las celdas de autocomprobación de los notebooks anteriores: una verificación automática vale más que una revisión apurada.

## Autocomprobación

In [ ]:
assert os.path.isdir("mi_informe/resultados/figuras"), "falta la estructura de carpetas"
assert os.path.exists("mi_informe/resultados/entorno.txt"), "falta el registro del entorno"
assert not faltan, "el portafolio de ejemplo está incompleto"
assert plt.rcParams["savefig.dpi"] == 300, "el estilo común no quedó aplicado"
print("Todo correcto.")

## Ejercicios

**1.** Añada a `registro_de_entorno()` la versión de `scikit-learn` y de `py3Dmol`, protegidas con `try/except` por si no están instaladas.

<details><summary><b>Ver solución</b></summary>

```python
for nombre in ["sklearn", "py3Dmol"]:
    try:
        mod = __import__(nombre)
        print(nombre, getattr(mod, "__version__", "sin versión"))
    except ImportError:
        print(nombre, "no instalado")
```
</details>

**2.** Escriba una función `pie_de_figura(numero, texto)` que devuelva el pie en el formato `**Figura 3.** texto` y úsela para las figuras de su informe.

<details><summary><b>Ver solución</b></summary>

```python
def pie_de_figura(numero, texto):
    return f"**Figura {numero}.** {texto}"

print(pie_de_figura(1, "Curva de calibración del analito, n = 3 por nivel."))
```
</details>

**3.** Complete la lista `ESPERADOS` con los archivos que produjo en las prácticas anteriores, con su nombre exacto y verifique la carpeta.

<details><summary><b>Ver solución</b></summary>

La estructura es:
```python
ESPERADOS = [f"mi_informe/resultados/figuras/fig{n:02d}_*.png" for n in range(1, 5)]
```
Con `glob.glob` se puede comprobar un patrón en vez de un nombre exacto.
</details>

**4.** Escriba un `README.md` dentro de `mi_informe/` que explique en diez líneas qué contiene la carpeta y en qué orden hay que correr las cosas.

<details><summary><b>Ver solución</b></summary>

```python
readme = """# Informe

Autor: ...

- `datos/`       datos de partida, sin modificar
- `resultados/`  salidas del código (tablas y figuras)
- `informe.ipynb` corre de principio a fin y regenera todo `resultados/`

Para reproducir: abrir informe.ipynb en Colab y ejecutar todas las celdas en orden.
Versiones usadas: ver resultados/entorno.txt
"""
open("mi_informe/README.md", "w", encoding="utf-8").write(readme)
```
</details>